# Work Orders — Part 2: A Worked Example (Task-Based)

This is the second notebook in the series that started with
`01-fhir-search-basics.ipynb`. Where Part 1 covered FHIR search syntax in
the abstract, this one works through one concrete, end-to-end scenario:
finding a laboratory's current work orders — genomic test orders
awaiting action by the lab — the same way this app's `/work-orders`
screen does (see `fhir_client.py`'s `active_filler_tasks()` and
`resolve_task_focus_order()`, and CLAUDE.md's "Work orders
(`/work-orders`)" section).

This app moved `/work-orders` from `ServiceRequest`-based filtering to
`Task`-based filtering — a `Task` here represents one unit of lab work
("please perform this genomic test"), separate from (but linked to) the
`ServiceRequest` that's the actual clinical order. This notebook follows
that same Task-first path:

1. Find the laboratory's `Organization` id (Liverpool GLH, ODS `K1S6S`).
2. Find its current work orders — `Task` resources with `status=requested`,
   `intent=filler-order`, owned by that Organization.
3. Retrieve one order in full — the `ServiceRequest`, its `Specimen`(s),
   and the `Patient` — and look at what comes back, including the Order
   Filler Number and specimen identifiers.
4. Handle the case where the patient being tested is a fetus or baby —
   who won't have an NHS number yet — and find the mother's own
   demographics via a `RelatedPerson` query.
5. Walk through the `Task.status` lifecycle the lab is expected to drive
   the order through: `requested` → `accepted`/`rejected`/`cancelled` →
   (eventually) `completed`.

Same ground rules as Part 1: every "build URL" cell runs with no server
access needed. Cells marked **live** actually call the FHIR server and are
guarded to skip cleanly if `FHIR_USER`/`FHIR_PASSWORD` aren't set — see
Part 1's own explanation of that convention. **Section 5's status-update
example is illustrative only and is never executed by this notebook** —
changing a real Task's status is a write with real workflow consequences,
not something a teaching notebook should do to production data as a side
effect of being run.

In [10]:
from urllib.parse import urlencode
from pathlib import Path
import os

# Load the repo's .env file (FHIR_BASE_URL, FHIR_USER, FHIR_PASSWORD, ...)
# into os.environ — same setup as 01-fhir-search-basics.ipynb (this
# notebook is self-contained, so it repeats it rather than depending on
# that one having been run first).
try:
    from dotenv import load_dotenv
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        env_path = candidate / ".env"
        if env_path.exists():
            load_dotenv(env_path)
            print(f"Loaded environment variables from {env_path}")
            break
    else:
        print("No .env file found in this directory or any parent — relying on "
              "already-exported environment variables, if any.")
except ImportError:
    print("python-dotenv isn't installed, so .env won't be loaded automatically. "
          "Run `pip install python-dotenv`, or export FHIR_BASE_URL/FHIR_USER/"
          "FHIR_PASSWORD yourself before starting Jupyter.")

# Matches CLAUDE.md's example deployment. Overridden by .env's FHIR_BASE_URL
# (or the FHIR_BASE_URL env var directly) if either is set.
BASE_URL = os.environ.get("FHIR_BASE_URL", "https://192.168.1.62/healthconnect/cdr/fhir/r4")


def fhir_search_url(resource_type, params):
    """Build a FHIR search URL the same shape fhir_client.py's _get() does.
    See 01-fhir-search-basics.ipynb for the full explanation of `doseq`."""
    query = urlencode(params, doseq=True)
    return f"{BASE_URL}/{resource_type}?{query}"


def fhir_read_url(resource_type, resource_id):
    """A direct FHIR *read* — GET [base]/[ResourceType]/[id] — returns the
    resource itself, not a searchset Bundle. Contrast with fhir_search_url()
    above; see section 3 below for why the distinction matters here."""
    return f"{BASE_URL}/{resource_type}/{resource_id}"


print(fhir_search_url("Organization", {"identifier": "K1S6S", "_count": 20}))

Loaded environment variables from /Users/kevinmayfield/github/MFT/julius/.env
https://192.168.1.62/healthconnect/cdr/fhir/r4/Organization?identifier=K1S6S&_count=20


## 1. Find the Organisation id — Liverpool GLH (ODS `K1S6S`)

Same `Organization?identifier=` search Part 1 covered (its section 5) —
an ODS code is just an `Organization.identifier` value, searched by bare
value for the same reason given there (the exact system URI a real ODS
code is tagged with on this server is unconfirmed).

**How `K1S6S` relates to `699X0`:** this app already has a second ODS
code on file for the North West Genomic Laboratory Hub network —
`FhirClient.ORDER_MESSAGE_DESTINATION_ODS = "699X0"`, display "NORTH WEST
GLH" — used as the fixed `destination` for every order message this app
builds (see CLAUDE.md's "Order creation" section: every order this app
sends goes to the same place, because this app is specifically for the
North West GLH). `699X0` is the **network/hub-level** ODS code — the one
registered for the North West GLH as a whole, used for routing and
commissioning. `K1S6S` is a **delivery-site-level** code underneath it —
Liverpool Clinical Laboratories' own registration, for the specific site
that actually receives and works these filler-order Tasks. This is a
normal NHS ODS pattern: a regional/network organisation has its own code,
and each member site that actually does the work keeps a separate code of
its own.

In FHIR terms, if this server models that parent/child relationship at
all, it would normally show up as `Organization.partOf` on the `K1S6S`
Organization, referencing the `699X0` Organization. **This is unconfirmed
against this real server** (same "detect, don't assume" caveat as
everything else in this codebase not yet checked against live data — see
CLAUDE.md's "Things that are unverified") — the live cell below checks
for it directly and reports what it actually finds.

**Postman:**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/Organization?identifier=K1S6S&_count=20
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

In [11]:
import requests
import json as _json

FHIR_USER = os.environ.get("FHIR_USER")
FHIR_PASSWORD = os.environ.get("FHIR_PASSWORD")

ods_code = "K1S6S"  # Liverpool GLH

liverpool_org = None
liverpool_org_id = None

if not (FHIR_USER and FHIR_PASSWORD):
    print("Set FHIR_USER / FHIR_PASSWORD env vars to try a live call. Skipping.")
else:
    resp = requests.get(
        fhir_search_url("Organization", {"identifier": ods_code, "_count": 20}),
        auth=(FHIR_USER, FHIR_PASSWORD),
        headers={"Accept": "application/fhir+json"},
        verify=os.environ.get("FHIR_VERIFY_SSL", "false").lower() == "true",
        timeout=30,
    )
    resp.raise_for_status()
    orgs = [
        entry["resource"] for entry in resp.json().get("entry", [])
        if entry.get("resource", {}).get("resourceType") == "Organization"
    ]
    print(f"{len(orgs)} Organization match(es) for ODS code {ods_code}")
    for org in orgs:
        print(f" - id={org['id']}  name={org.get('name')!r}")

    if orgs:
        liverpool_org = orgs[0]
        liverpool_org_id = liverpool_org["id"]
        print()
        print(f"Using Organization/{liverpool_org_id} as the owner for section 2 below.")

        # Check whether this server models the 699X0 (North West GLH) /
        # K1S6S (Liverpool GLH site) relationship via Organization.partOf.
        part_of = liverpool_org.get("partOf")
        if not part_of:
            print("No Organization.partOf on this resource — the hub/site "
                  "relationship (if this server models it at all) isn't "
                  "expressed this way here.")
        else:
            print(f"Organization.partOf: {part_of}")
            parent = None
            if part_of.get("reference"):
                parent_resp = requests.get(
                    f"{BASE_URL}/{part_of['reference']}",
                    auth=(FHIR_USER, FHIR_PASSWORD),
                    headers={"Accept": "application/fhir+json"},
                    verify=os.environ.get("FHIR_VERIFY_SSL", "false").lower() == "true",
                    timeout=30,
                )
                if parent_resp.ok:
                    parent = parent_resp.json()
            if parent:
                parent_ods = [ident.get("value") for ident in parent.get("identifier", [])]
                print(f"Parent Organization: name={parent.get('name')!r}  identifiers={parent_ods}")
                if "699X0" in parent_ods:
                    print("Confirmed: parent is the 699X0 North West GLH network Organization.")
                else:
                    print("Parent's ODS code(s) don't include 699X0 — check manually.")
            else:
                print("partOf reference didn't resolve to a fetchable Organization.")

1 Organization match(es) for ODS code K1S6S
 - id=21567  name='NORTH WEST GENOMIC LABORATORY HUB LIVERPOOL'

Using Organization/21567 as the owner for section 2 below.
Organization.partOf: {'identifier': {'system': 'https://fhir.nhs.uk/Id/ods-organization-code', 'value': 'QOP'}, 'reference': 'Organization/32'}
Parent Organization: name='NHS GREATER MANCHESTER INTEGRATED CARE BOARD'  identifiers=['QOP']
Parent's ODS code(s) don't include 699X0 — check manually.


/Users/kevinmayfield/github/MFT/julius/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/kevinmayfield/github/MFT/julius/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## 2. Find work orders — `Task` resources for this owner

This is exactly the query `FhirClient.active_filler_tasks()` builds for
the `/work-orders` screen (see CLAUDE.md's "Work orders (`/work-orders`)"
section): `Task.intent=filler-order`, owned by the Organization found
above, filtered by `status` (defaulting to `"requested"` — the screen's
own `FhirClient.DEFAULT_TASK_STATUS`).

Two things worth calling out that are genuinely different from the
ServiceRequest-based searches in Part 1:

- **`owner` is a reference search parameter** — same shape as
  `performer` in Part 1's section 7 — matching `Task.owner`. You need the
  Organization's *id*, not its ODS code, which is exactly why section 1
  above came first.
- **This query is deliberately scoped by owner, not fetched
  system-wide.** An unscoped `Task?status=requested&intent=filler-order`
  query across every laboratory on the server would risk exactly the 413
  behaviour documented in CLAUDE.md's "413s on unfiltered system-wide
  searches" — Task is new to this app and untested at real volume, so
  `active_filler_tasks()` takes the same precaution every other
  system-wide query in `fhir_client.py` already does.

The `_include`/`_include:iterate` parameters below match
`TASK_INCLUDES`/`TASK_ITERATE_INCLUDES` in `fhir_client.py` — they pull
the Task's patient, focus order, owner, and requester back in the same
Bundle (plus, one hop further, the Practitioner/Organization behind a
PractitionerRole, and the focus order's own Specimen), the same
`_include` mechanism Part 1 introduced at the end of its section 11.

In [12]:
# Matches TASK_INCLUDES / TASK_ITERATE_INCLUDES in fhir_client.py.
TASK_INCLUDES = ["Task:patient", "Task:focus", "Task:owner", "Task:requester"]
TASK_ITERATE_INCLUDES = ["PractitionerRole:practitioner", "PractitionerRole:organization", "ServiceRequest:specimen"]

owner_organization_id = liverpool_org_id or "<organization-id-from-section-1>"

url = fhir_search_url("Task", {
    "status": "requested",
    "intent": "filler-order",
    "owner": f"Organization/{owner_organization_id}",
    "_count": 100,
    "_include": TASK_INCLUDES,
    "_include:iterate": TASK_ITERATE_INCLUDES,
})
print(url)

https://192.168.1.62/healthconnect/cdr/fhir/r4/Task?status=requested&intent=filler-order&owner=Organization%2F21567&_count=100&_include=Task%3Apatient&_include=Task%3Afocus&_include=Task%3Aowner&_include=Task%3Arequester&_include%3Aiterate=PractitionerRole%3Apractitioner&_include%3Aiterate=PractitionerRole%3Aorganization&_include%3Aiterate=ServiceRequest%3Aspecimen


**Postman:**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/Task?status=requested&intent=filler-order&owner=Organization/<id>&_count=100&_include=Task:patient&_include=Task:focus&_include=Task:owner&_include=Task:requester&_include:iterate=PractitionerRole:practitioner&_include:iterate=PractitionerRole:organization&_include:iterate=ServiceRequest:specimen
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

In [13]:
tasks = []
first_task = None

if not (FHIR_USER and FHIR_PASSWORD):
    print("Set FHIR_USER / FHIR_PASSWORD env vars to try a live call. Skipping.")
elif not liverpool_org_id:
    print("No Organization id from section 1 to search Task.owner against. Skipping.")
else:
    resp = requests.get(
        fhir_search_url("Task", {
            "status": "requested",
            "intent": "filler-order",
            "owner": f"Organization/{liverpool_org_id}",
            "_count": 100,
            "_include": TASK_INCLUDES,
            "_include:iterate": TASK_ITERATE_INCLUDES,
        }),
        auth=(FHIR_USER, FHIR_PASSWORD),
        headers={"Accept": "application/fhir+json"},
        verify=os.environ.get("FHIR_VERIFY_SSL", "false").lower() == "true",
        timeout=30,
    )
    resp.raise_for_status()
    bundle = resp.json()
    entries = bundle.get("entry", [])
    # Same caveat as ctdna_orders()/_active_orders_with_intent() in
    # fhir_client.py: identify Tasks by resourceType across every entry
    # rather than trusting entry.search.mode, in case this server doesn't
    # tag it reliably on _include'd resources.
    tasks = [e["resource"] for e in entries if e.get("resource", {}).get("resourceType") == "Task"]
    print(f"{len(tasks)} requested filler-order Task(s) owned by Organization/{liverpool_org_id}")
    for t in tasks:
        print(f" - id={t['id']}  status={t.get('status')}  intent={t.get('intent')}  "
              f"authoredOn={t.get('authoredOn')}  focus={(t.get('focus') or {}).get('reference')}")

    if tasks:
        first_task = tasks[0]
        print()
        print(f"Using Task/{first_task['id']} for section 3 below.")

38 requested filler-order Task(s) owned by Organization/21567
 - id=207396  status=requested  intent=filler-order  authoredOn=2026-08-26T07:22:05Z  focus=ServiceRequest/206226
 - id=207398  status=requested  intent=filler-order  authoredOn=2026-08-26T07:22:05Z  focus=ServiceRequest/206253
 - id=207400  status=requested  intent=filler-order  authoredOn=2026-08-26T07:22:06Z  focus=ServiceRequest/206257
 - id=207402  status=requested  intent=filler-order  authoredOn=2026-08-26T07:22:06Z  focus=ServiceRequest/206261
 - id=207404  status=requested  intent=filler-order  authoredOn=2026-08-26T07:22:06Z  focus=ServiceRequest/206269
 - id=207406  status=requested  intent=filler-order  authoredOn=2026-08-26T07:22:06Z  focus=ServiceRequest/206309
 - id=207408  status=requested  intent=filler-order  authoredOn=2026-08-26T07:22:06Z  focus=ServiceRequest/206338
 - id=207410  status=requested  intent=filler-order  authoredOn=2026-08-26T07:22:06Z  focus=ServiceRequest/206359
 - id=207412  status=reque

/Users/kevinmayfield/github/MFT/julius/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## 3. Retrieve the order — `ServiceRequest`, `Specimen`, and `Patient`

`Task.focus` is a reference straight to the `ServiceRequest` this Task
was created to fulfil — this app's `resolve_task_focus_order()` follows
exactly this reference (falling back to `None` if it's absent or doesn't
resolve to a `ServiceRequest`, since this deployment's Task usage is new
enough that neither is guaranteed — see that method's docstring in
`fhir_client.py`).

Unlike every search so far, following `Task.focus` / `ServiceRequest.
specimen` / `ServiceRequest.subject` are plain FHIR **reads**, not
searches:

```
GET [base]/[ResourceType]/[id]
```

A read returns the resource itself directly — no `Bundle` wrapper, no
`entry[]`, just the JSON you'd expect. That's the one structural
difference from every example in Part 1 and sections 1–2 above, all of
which searched and got a `searchset` Bundle back.

**Fields worth looking at once the ServiceRequest comes back:**

- **Order Filler Number** — `ServiceRequest.identifier[]`, the entry
  typed HL7 v2-0203 code `"FILL"` (`FhirClient.FILLER_IDENTIFIER_TYPE`,
  read back by `filler_identifier()`) — the identifier the *lab* assigned
  to this order, as opposed to a `"PLAC"`-typed Placer Order Number
  (`placer_identifier()`) assigned by whoever originally requested it.
- **Specimen identifiers** — `ServiceRequest.specimen[]` is a list of
  references (`resolve_specimens()` resolves all of them); each
  `Specimen.identifier[]` can carry more than one flavour per the
  Specimen profile's own Domain Archetype (see CLAUDE.md's "Order
  creation" → "Specimen is mandatory" section) — a Placer Specimen
  Number, and a Shipment Tracking Number (LOINC `97209-1`) — plus a
  separate `Specimen.accessionIdentifier` for the lab's own accession
  number once it's logged the sample in. `Specimen.type` is a SNOMED
  coding (from the IG's `specimen-type` ValueSet) naming what kind of
  sample it physically is (blood, saliva, tissue, ...).
- **The Patient** — `ServiceRequest.subject` — is where section 4 below
  picks up.

In [14]:
example_service_request_id = ((first_task or {}).get("focus") or {}).get("reference", "").split("/")[-1] \
    or "<service-request-id-from-section-2>"
print(fhir_read_url("ServiceRequest", example_service_request_id))

https://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest/206226


**Postman:**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/ServiceRequest/<id>
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

(and the same shape — `GET .../Specimen/<id>`, `GET .../Patient/<id>` —
for the specimen(s) and patient once you have their ids from the
ServiceRequest below.)

In [15]:
focus_order = None
specimens = []
order_patient = None


def fhir_read(resource_type, resource_id):
    resp = requests.get(
        fhir_read_url(resource_type, resource_id),
        auth=(FHIR_USER, FHIR_PASSWORD),
        headers={"Accept": "application/fhir+json"},
        verify=os.environ.get("FHIR_VERIFY_SSL", "false").lower() == "true",
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()


def identifier_by_type(resource, type_code):
    for ident in resource.get("identifier", []):
        codes = [c.get("code") for c in (ident.get("type") or {}).get("coding", [])]
        if type_code in codes:
            return ident
    return None


if not (FHIR_USER and FHIR_PASSWORD):
    print("Set FHIR_USER / FHIR_PASSWORD env vars to try a live call. Skipping.")
elif not first_task:
    print("No Task from section 2 to follow .focus from. Skipping.")
elif not (first_task.get("focus") or {}).get("reference"):
    print(f"Task/{first_task['id']} has no .focus reference to follow.")
else:
    focus_ref = first_task["focus"]["reference"]  # e.g. "ServiceRequest/12345"
    focus_type, focus_id = focus_ref.split("/", 1)
    focus_order = fhir_read(focus_type, focus_id)

    filler = identifier_by_type(focus_order, "FILL")
    placer = identifier_by_type(focus_order, "PLAC")
    print(f"ServiceRequest/{focus_order['id']}  status={focus_order.get('status')}  "
          f"intent={focus_order.get('intent')}")
    print(f"Order Filler Number: {filler.get('value') if filler else '(none)'}")
    print(f"Order Placer Number: {placer.get('value') if placer else '(none)'}")

    for ref in focus_order.get("specimen", []):
        if not ref.get("reference"):
            continue
        spec_type, spec_id = ref["reference"].split("/", 1)
        specimen = fhir_read(spec_type, spec_id)
        specimens.append(specimen)
        spec_type_display = ((specimen.get("type") or {}).get("coding") or [{}])[0].get("display")
        print()
        print(f"Specimen/{specimen['id']}  type={spec_type_display}")
        for ident in specimen.get("identifier", []):
            ident_type = ident.get("type") or {}
            label = (ident_type.get("text")
                     or next((c.get("display") for c in ident_type.get("coding", [])), None)
                     or "identifier")
            print(f"  - {label}: {ident.get('value')}")
        if specimen.get("accessionIdentifier"):
            print(f"  - Accession number: {specimen['accessionIdentifier'].get('value')}")

    subject_ref = (focus_order.get("subject") or {}).get("reference")
    if subject_ref:
        pat_type, pat_id = subject_ref.split("/", 1)
        order_patient = fhir_read(pat_type, pat_id)
        name = (order_patient.get("name") or [{}])[0]
        display_name = " ".join([*name.get("given", []), name.get("family", "")]).strip()
        nhs = next((i.get("value") for i in order_patient.get("identifier", [])
                    if i.get("system") == "https://fhir.nhs.uk/Id/nhs-number"), None)
        print()
        print(f"Patient/{order_patient['id']}  name={display_name or '(no name)'}  "
              f"birthDate={order_patient.get('birthDate')}  NHS number={nhs or '(none)'}")

/Users/kevinmayfield/github/MFT/julius/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


ServiceRequest/206226  status=active  intent=filler-order
Order Filler Number: T26-6V04
Order Placer Number: (none)

Specimen/206224  type=Fresh tissue specimen
  - identifier: S26-1ZZ4

Patient/200770  name=Baby of Lysa BUXTON  birthDate=1971-01-01  NHS number=(none)


/Users/kevinmayfield/github/MFT/julius/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/kevinmayfield/github/MFT/julius/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## 4. When the patient is a fetus or baby — no NHS number, and finding the mother

Some genomic tests in this IG are prenatal (testing a **fetus**, ahead of
birth) or neonatal (testing a newborn **baby**) — a rare disease panel
requested urgently for a sick neonate is a common real example. A
`Patient` resource for a fetus/baby often **doesn't have an NHS number
yet** — a number may not have been traced/issued at the point testing is
requested (see CLAUDE.md's "Patient matching": NHS number is preferred
but this app always treats it as *available*, not *guaranteed*).

Instead, this kind of Patient is identified by whatever the requesting
organisation already uses internally:

- **Medical Record Number** — HL7 v2-0203 type `"MR"`
  (`FhirClient.MEDICAL_RECORD_NUMBER_TYPE`, read back by
  `medical_record_numbers()`) — the hospital's own patient number, one
  per assigning organisation.
- **Patient Identifier** — HL7 v2-0203 type `"PI"`
  (`FhirClient.PATIENT_IDENTIFIER_TYPE`, read back by
  `igene_patient_identifier()`) — an internal identifier from the lab's
  own iGene system.

You'd recognise this case in the data above (section 3) as a `Patient`
whose `identifier[]` has no `https://fhir.nhs.uk/Id/nhs-number`-system
entry, but does have one typed `"MR"` and/or `"PI"`.

**Finding the mother**: FHIR models this via `RelatedPerson` — a resource
for someone who matters to a patient's record without necessarily being a
fully registered patient themselves. `RelatedPerson.patient` references
the fetus/baby (the Patient we're interested in), but the **mother's own
demographics — name, date of birth, contact details — are carried
directly on the `RelatedPerson` resource itself**, not via a second
reference to a separate Patient record for her. `RelatedPerson.
relationship` carries a role code — the maternal role would normally be
HL7 v3 RoleCode `"MTH"` ("mother") or `"NMTH"` ("natural mother"), though
**which code this server actually uses is unconfirmed** — check
`relationship.coding[].code` directly on whatever comes back below.

This app already has the query this needs —
`FhirClient.related_persons_for_patient(patient_id)` does exactly
`RelatedPerson?patient=<id>` — used today by the patient page's
clear-down feature; it isn't yet surfaced as its own screen, but the
query itself is already there to reuse.

In [16]:
example_patient_id = (order_patient or {}).get("id") or "<patient-id-from-section-3>"

url = fhir_search_url("RelatedPerson", {"patient": example_patient_id, "_count": 20})
print(url)

https://192.168.1.62/healthconnect/cdr/fhir/r4/RelatedPerson?patient=200770&_count=20


**Postman:**

```
GET https://192.168.1.62/healthconnect/cdr/fhir/r4/RelatedPerson?patient=<patient-id>&_count=20
Accept: application/fhir+json
Authorization: Basic Auth (your username/password)
```

In [17]:
related_persons = []

if not (FHIR_USER and FHIR_PASSWORD):
    print("Set FHIR_USER / FHIR_PASSWORD env vars to try a live call. Skipping.")
elif not order_patient:
    print("No Patient from section 3 to search RelatedPerson against. Skipping.")
else:
    resp = requests.get(
        fhir_search_url("RelatedPerson", {"patient": order_patient["id"], "_count": 20}),
        auth=(FHIR_USER, FHIR_PASSWORD),
        headers={"Accept": "application/fhir+json"},
        verify=os.environ.get("FHIR_VERIFY_SSL", "false").lower() == "true",
        timeout=30,
    )
    resp.raise_for_status()
    related_persons = [
        e["resource"] for e in resp.json().get("entry", [])
        if e.get("resource", {}).get("resourceType") == "RelatedPerson"
    ]
    print(f"{len(related_persons)} RelatedPerson(s) for Patient/{order_patient['id']}")

    nhs = next((i.get("value") for i in order_patient.get("identifier", [])
                if i.get("system") == "https://fhir.nhs.uk/Id/nhs-number"), None)
    mrns = [i.get("value") for i in order_patient.get("identifier", [])
            if any(c.get("code") == "MR" for c in (i.get("type") or {}).get("coding", []))]
    pis = [i.get("value") for i in order_patient.get("identifier", [])
           if any(c.get("code") == "PI" for c in (i.get("type") or {}).get("coding", []))]
    print(f"This patient's NHS number: {nhs or '(none — possibly a fetus/baby case)'}")
    print(f"Medical Record Number(s): {mrns or '(none)'}")
    print(f"Patient Identifier(s) (PI): {pis or '(none)'}")

    for rp in related_persons:
        name = (rp.get("name") or [{}])[0]
        display_name = " ".join([*name.get("given", []), name.get("family", "")]).strip()
        rel_codes = [c.get("code") for r in rp.get("relationship", []) for c in r.get("coding", [])]
        print()
        print(f"RelatedPerson/{rp['id']}  relationship={rel_codes or '(none)'}  "
              f"name={display_name or '(no name)'}  birthDate={rp.get('birthDate')}")

/Users/kevinmayfield/github/MFT/julius/.venv/lib/python3.10/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.1.62'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


20 RelatedPerson(s) for Patient/200770
This patient's NHS number: (none — possibly a fetus/baby case)
Medical Record Number(s): ['QE1200083']
Patient Identifier(s) (PI): ['200083', '200083']

RelatedPerson/200775  relationship=['MTH']  name=Lysa BUXTON  birthDate=None

RelatedPerson/200864  relationship=['MTH']  name=Lysa BUXTON  birthDate=1971-01-01

RelatedPerson/200865  relationship=['MTH']  name=Lysa BUXTON  birthDate=None

RelatedPerson/200882  relationship=['MTH']  name=Lysa BUXTON  birthDate=1971-01-01

RelatedPerson/201450  relationship=['MTH']  name=Lysa BUXTON  birthDate=1971-01-01

RelatedPerson/201559  relationship=['MTH']  name=Lysa BUXTON  birthDate=None

RelatedPerson/202204  relationship=['MTH']  name=Lysa BUXTON  birthDate=1971-01-01

RelatedPerson/202209  relationship=['MTH']  name=Lysa BUXTON  birthDate=1971-01-01

RelatedPerson/202365  relationship=['MTH']  name=Lysa BUXTON  birthDate=1971-01-01

RelatedPerson/202821  relationship=['MTH']  name=Lysa BUXTON  birthDat

## 5. Taking ownership — updating `Task.status`

Once a lab has actually found and reviewed a `requested` Task like the
one above, normal workflow is for it to move that Task's status to show
what it decided:

- **`accepted`** — the lab has checked the request (and specimen) and
  will proceed with testing.
- **`rejected`** — the lab has found a fault (e.g. an unsuitable
  specimen, a request it can't fulfil) and won't proceed.
- **`cancelled`** — the order itself is being withdrawn rather than
  actively refused.

`Task.statusReason` (a `CodeableConcept`, separate from `status` itself)
is where the *why* goes — particularly worth populating on a `rejected`/
`cancelled` transition, so whoever placed the order can see why without
having to ask.

**This is a write, and this app doesn't have a button for it yet** — per
CLAUDE.md, the only two places this app currently writes to a FHIR server
at all are `scripts/fix_organization_names.py` (backfilling an
Organization's name) and `/order/new`'s "Send to ESB" button (sending a
brand-new order message) — nothing today updates an existing Task. The
shape such an update *would* take, though, mirrors
`update_organization_name()`'s existing `_put()` pattern in
`fhir_client.py`: fetch the resource, change only the field(s) that
changed, `PUT` the whole thing back.

**`If-Match` matters here more than it did for that Organization-name
fix**: a Task can plausibly be looked at by more than one person, and a
straight `PUT` with no version check would silently overwrite anything
someone else changed in between your `GET` and your `PUT`. Sending
`If-Match: W/"<versionId>"` (from the Task's own `meta.versionId`) makes
the `PUT` fail with `412 Precondition Failed` instead of overwriting a
concurrent change — a real risk this deployment doesn't need to
special-case, since it's exactly what that header exists for.

The cell below **builds the example update but never sends it** —
mutating a real Task's status is a genuine workflow action with real
consequences downstream (a lab that "accepts" an order commits to acting
on it), not something a teaching notebook should trigger as a side
effect of being run through. Nothing in this cell makes an HTTP call.

In [18]:
def build_task_status_update(task, new_status, reason_text=None):
    """Build the (unsent) PUT body/headers for moving `task` to
    `new_status` — mirrors update_organization_name()'s fetch-modify-PUT
    pattern in fhir_client.py, plus an If-Match header built from the
    Task's own meta.versionId for optimistic concurrency.

    Returns (url, headers, body). Nothing is sent — see the markdown above.
    """
    updated = dict(task)  # shallow copy — don't mutate the caller's Task
    updated["status"] = new_status
    if reason_text:
        updated["statusReason"] = {"text": reason_text}

    version_id = (task.get("meta") or {}).get("versionId")
    headers = {"Content-Type": "application/fhir+json"}
    if version_id:
        headers["If-Match"] = f'W/"{version_id}"'

    url = fhir_read_url("Task", task["id"])
    return url, headers, updated


example_task = first_task or {"id": "<task-id-from-section-2>", "meta": {"versionId": "<version-id>"}}

# Accepting the order:
url, headers, body = build_task_status_update(example_task, "accepted")
print(f"PUT {url}")
for k, v in headers.items():
    print(f"{k}: {v}")
print(_json.dumps({"status": body["status"]}, indent=2), " # (+ every other field unchanged)")

print()

# Rejecting it instead, with a reason:
url, headers, body = build_task_status_update(
    example_task, "rejected",
    reason_text="Specimen received in an unsuitable condition for genomic testing.",
)
print(f"PUT {url}")
for k, v in headers.items():
    print(f"{k}: {v}")
print(_json.dumps({"status": body["status"], "statusReason": body.get("statusReason")}, indent=2),
      " # (+ every other field unchanged)")

PUT https://192.168.1.62/healthconnect/cdr/fhir/r4/Task/207396
Content-Type: application/fhir+json
If-Match: W/"1"
{
  "status": "accepted"
}  # (+ every other field unchanged)

PUT https://192.168.1.62/healthconnect/cdr/fhir/r4/Task/207396
Content-Type: application/fhir+json
If-Match: W/"1"
{
  "status": "rejected",
  "statusReason": {
    "text": "Specimen received in an unsuitable condition for genomic testing."
  }
}  # (+ every other field unchanged)


**Postman (illustrative — do not send against a real Task unless you
mean to change its status):**

```
PUT [base]/Task/<task-id>
Content-Type: application/fhir+json
If-Match: W/"<version-id>"
Authorization: Basic Auth (your username/password)

{
  ...every existing field on the Task, unchanged...,
  "status": "accepted"
}
```

A full-resource `PUT` is what this app's own `_put()` helper already
does elsewhere (`update_organization_name()`), so it's the path of least
resistance if an "accept this Task" button gets built later. If a server
supports FHIR's `PATCH` verb, a lighter alternative only touches the
field that's actually changing, without resending the whole resource:

```
PATCH [base]/Task/<task-id>
Content-Type: application/json-patch+json
If-Match: W/"<version-id>"
Authorization: Basic Auth (your username/password)

[
  {"op": "replace", "path": "/status", "value": "accepted"}
]
```

**`PATCH` support is not something to assume.** FHIR's base spec makes
`PATCH` optional, and plenty of off-the-shelf FHIR server products —
including, quite possibly, this deployment's own InterSystems IRIS-backed
CDR (see CLAUDE.md's "413s on unfiltered system-wide searches" for other
behaviour specific to this server that had to be discovered rather than
assumed) — only implement `GET`/`POST`/`PUT`/`DELETE` and reject `PATCH`
outright (commonly a `405 Method Not Allowed`), or only support one of
the three PATCH content types FHIR allows (JSON Patch, as above; XML
Patch; or FHIRPath Patch via a `Parameters` resource) rather than all
three. Check for a real `2xx` response the first time you try it against
a given server before relying on it — the full-resource `PUT` above is
the safer default this notebook leads with precisely because every FHIR
server is required to support it.

## 6. The lifecycle doesn't stop at `accepted`

`accepted` only means the lab has taken the work on — it isn't the last
status change a Task normally goes through. FHIR R4's own `task-status`
ValueSet (`FhirClient.TASK_STATUS_VALUES` in `fhir_client.py` — the same
list the work orders screen's status filter is built from) models the
rest of the journey too: once the lab actually starts the genomic
testing, the Task would move to `in-progress`, and only once that testing
is actually finished — the point a `DiagnosticReport` normally exists,
linked back to the original order via `DiagnosticReport.basedOn` (see
CLAUDE.md's ctDNA/Cepheid sections for how this app already follows that
same link the other direction) — does it reach `completed`, the genuine
terminal state for a successful order.

```
requested → accepted → in-progress → completed
              │
              ├──→ rejected     (fault found; won't proceed)
              └──→ cancelled    (order withdrawn)
```

The work orders screen's own status filter (`/work-orders?status=...`,
defaulting to `requested`) exists precisely so a lab can look at any one
of these states on demand — not just the freshly-arrived `requested`
queue this notebook started with.

## What's next

This notebook worked through one concrete Task-based scenario end to end
— finding a lab's queue, retrieving the underlying order/specimen/
patient, handling the fetus/baby-without-an-NHS-number case via
RelatedPerson, and the status lifecycle a lab is expected to drive an
order through. Part 1's original "what's next" list still holds for what
comes after this:

- **Category-coded searches and status/status-reason filtering** — the
  try-categorized-then-fall-back pattern this app uses throughout (see
  "Category codes come from the IG, not guesses" in CLAUDE.md).
- **Pagination and the 413 story** — why an unscoped system-wide search
  can fail on a real server, and the organisation-scoped batching pattern
  this app had to adopt (CLAUDE.md's "413s on unfiltered system-wide
  searches" — the same reasoning `active_filler_tasks()` in this notebook
  already leaned on).
- **Writing data for real** — this notebook deliberately stopped short of
  sending the Task status update in section 5. A future notebook could
  cover building and sending a write for real, the way `/order/new`'s
  "Send to ESB" button already does for brand-new orders (see
  `FhirClient.send_order_to_esb()`), including what to check for in the
  response and how to handle a `412 Precondition Failed` from a stale
  `If-Match`.